# Notebook 4 — Advanced Connectome Analysis and Visualization

**BINFX410 · Chapter 10 · Connectomics**

---

## Learning Objectives

1. Extract a structured **adjacency matrix** from the connectome graph
2. Analyze information flow using **random walk** simulations
3. Detect and visualize **motifs** (recurring sub-circuit patterns)
4. Perform **hierarchical clustering** of neurons based on connectivity profiles
5. Export connectome properties that will guide neural network design in Notebook 5

---

## Background

### Adjacency Matrix
The adjacency matrix $A$ of a graph on $N$ nodes is an $N \times N$ matrix where $A_{ij} = w_{ij}$ if there is an edge from node $i$ to node $j$, and 0 otherwise. For connectomes, $A_{ij}$ typically encodes **synapse count** or **connection strength**.

### Network Motifs
**Motifs** are small recurring subgraph patterns that appear more often than expected by chance. In neural circuits, common motifs include:
- **Feedforward loop** (A→B→C, A→C): enables signal filtering
- **Feedback loop** (A→B→A): enables oscillations and memory
- **Divergence** (A→B, A→C): one sender → many receivers
- **Convergence** (A→C, B→C): many senders → one receiver

### Connectivity Fingerprints
Each neuron can be characterized by its **connectivity fingerprint**: the vector of edge weights in its row and column of the adjacency matrix. Neurons with similar fingerprints perform similar roles in the circuit.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

DATA_DIR = Path('../data')

with open(DATA_DIR / 'connectome_graphs.pkl', 'rb') as f:
    data = pickle.load(f)

G  = data['G_detected']
GT = data['G_gt']
detected_soma = data['detected_soma']
image         = data['image']

nodes = sorted(G.nodes())
N     = len(nodes)
print(f'Connectome: {N} neurons, {G.number_of_edges()} synaptic connections')

## 1. Adjacency Matrix

In [ ]:
# Build dense adjacency matrix
A = nx.to_numpy_array(G, nodelist=nodes, weight='weight')

print(f'Adjacency matrix shape: {A.shape}')
print(f'Non-zero entries      : {(A > 0).sum()}')
print(f'Weight range          : [{A.min():.2f}, {A.max():.2f}]')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(A, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Synapse count')
ax.set_xlabel('Post-synaptic neuron (j)')
ax.set_ylabel('Pre-synaptic neuron (i)')
ax.set_title('Connectome Adjacency Matrix $A_{ij}$', fontsize=13)
ax.set_xticks(range(N))
ax.set_yticks(range(N))
ax.set_xticklabels([f'N{n}' for n in nodes], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([f'N{n}' for n in nodes], fontsize=8)
plt.tight_layout()
plt.show()

## 2. Normalized Connectivity and Information Flow

In [ ]:
# Row-normalize A to get a transition probability matrix P
row_sums = A.sum(axis=1, keepdims=True)
P = np.where(row_sums > 0, A / row_sums, 0)

# Simulate a random walk: start at each node, run for T steps
def random_walk_distribution(P, start_node, T=10):
    """Probability distribution over nodes after T random walk steps."""
    dist = np.zeros(len(P))
    dist[start_node] = 1.0
    for _ in range(T):
        dist = dist @ P
    return dist


# Influence matrix: I[i, j] = probability of reaching j starting from i in 5 steps
T_steps = 5
influence = np.vstack([random_walk_distribution(P, i, T=T_steps) for i in range(N)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(P, cmap='Oranges', aspect='auto')
plt.colorbar(im0, ax=axes[0], label='Transition probability')
axes[0].set_title('Transition Probability Matrix P', fontsize=12)
axes[0].set_xlabel('Target node j')
axes[0].set_ylabel('Source node i')

im1 = axes[1].imshow(influence, cmap='Oranges', aspect='auto')
plt.colorbar(im1, ax=axes[1], label=f'Reach probability ({T_steps} steps)')
axes[1].set_title(f'Influence Matrix ({T_steps}-step Random Walk)', fontsize=12)
axes[1].set_xlabel('Target node j')
axes[1].set_ylabel('Source node i')

plt.tight_layout()
plt.show()

## 3. Circuit Motif Detection

In [ ]:
def count_feedforward_loops(G):
    """Count 3-node feedforward loop motifs: A→B, B→C, A→C."""
    count = 0
    nodes_list = list(G.nodes())
    for a in nodes_list:
        for b in G.successors(a):
            for c in G.successors(b):
                if c != a and G.has_edge(a, c):
                    count += 1
    return count


def count_feedback_loops(G):
    """Count mutual connection pairs: A→B and B→A."""
    count = 0
    for a, b in G.edges():
        if G.has_edge(b, a):
            count += 1
    return count // 2  # each pair counted twice


def count_divergence(G, min_out=2):
    """Count nodes with out-degree >= min_out (divergence hubs)."""
    return sum(1 for n in G.nodes() if G.out_degree(n) >= min_out)


def count_convergence(G, min_in=2):
    """Count nodes with in-degree >= min_in (convergence hubs)."""
    return sum(1 for n in G.nodes() if G.in_degree(n) >= min_in)


ffl  = count_feedforward_loops(G)
fbl  = count_feedback_loops(G)
div  = count_divergence(G, min_out=2)
conv = count_convergence(G, min_in=2)

print('Detected Connectome Motifs')
print(f'  Feedforward loops (A→B→C, A→C) : {ffl}')
print(f'  Mutual connections  (A⇄B)       : {fbl}')
print(f'  Divergence hubs (out ≥ 2)       : {div}')
print(f'  Convergence hubs (in  ≥ 2)      : {conv}')

# Compare to a random graph baseline
n_rand_trials = 20
rand_ffl_counts = []
for seed in range(n_rand_trials):
    Gr = nx.gnm_random_graph(G.number_of_nodes(), G.number_of_edges(),
                              directed=True, seed=seed)
    rand_ffl_counts.append(count_feedforward_loops(Gr))

rand_mean = np.mean(rand_ffl_counts)
rand_std  = np.std(rand_ffl_counts)
print(f'\nRandom graph FFL (mean ± std): {rand_mean:.1f} ± {rand_std:.1f}')
print(f'Enrichment Z-score: {(ffl - rand_mean) / (rand_std + 1e-8):.2f}')

## 4. Connectivity Fingerprints and Hierarchical Clustering

In [ ]:
# Each neuron's fingerprint = its row in A (outgoing) concatenated with its column (incoming)
fingerprints = np.hstack([A, A.T])  # (N, 2N)

if N >= 3:
    # Hierarchical clustering
    dist_matrix = pdist(fingerprints, metric='euclidean')
    Z           = linkage(dist_matrix, method='ward')

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Dendrogram
    dn = dendrogram(Z, ax=axes[0],
                    labels=[f'N{n}' for n in nodes],
                    leaf_rotation=45, color_threshold=0.6 * max(Z[:, 2]))
    axes[0].set_title('Hierarchical Clustering of Neurons\n(by Connectivity Fingerprint)', fontsize=12)
    axes[0].set_ylabel('Ward Distance')

    # Reordered adjacency matrix
    order    = dn['leaves']
    A_reord  = A[np.ix_(order, order)]
    im = axes[1].imshow(A_reord, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=axes[1], label='Synapse count')
    reordered_labels = [f'N{nodes[i]}' for i in order]
    axes[1].set_xticks(range(N))
    axes[1].set_yticks(range(N))
    axes[1].set_xticklabels(reordered_labels, rotation=45, ha='right', fontsize=8)
    axes[1].set_yticklabels(reordered_labels, fontsize=8)
    axes[1].set_title('Adjacency Matrix (Clustered Order)', fontsize=12)

    plt.tight_layout()
    plt.show()
else:
    print('Not enough neurons for hierarchical clustering.')

## 5. Visualize Signal Propagation

In [ ]:
# Simulate signal spreading from the highest-PageRank neuron
if G.number_of_edges() > 0 and N >= 2:
    pagerank = nx.pagerank(G, alpha=0.85)
    seed_neuron = max(pagerank, key=pagerank.get)

    activation_over_time = []
    act = np.zeros(N)
    act[seed_neuron] = 1.0
    activation_over_time.append(act.copy())

    for t in range(6):
        act = act @ P
        activation_over_time.append(act.copy())

    # Visualize as a heatmap over time
    act_matrix = np.array(activation_over_time)  # (T, N)

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(act_matrix.T, cmap='YlOrRd', aspect='auto',
                   interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Activation')
    ax.set_yticks(range(N))
    ax.set_yticklabels([f'N{n}' for n in nodes])
    ax.set_xlabel('Time step')
    ax.set_title(f'Signal Propagation from Neuron N{seed_neuron} (highest PageRank)', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Find which neuron receives the signal most strongly at t=5
    final_act = activation_over_time[-1]
    top3 = np.argsort(final_act)[::-1][:3]
    print(f'Seed neuron: N{seed_neuron}')
    print('Top receivers after 6 steps:')
    for rank, n in enumerate(top3):
        print(f'  {rank + 1}. N{n}: {final_act[n]:.4f}')

## 6. Export Connectome Architecture for Neural Network Design

In [ ]:
# Identify layer-like structure via topological sort (if DAG) or BFS layers
G_simple = nx.DiGraph()
G_simple.add_nodes_from(G.nodes())
G_simple.add_edges_from(G.edges())

# Remove self-loops for DAG analysis
G_simple.remove_edges_from(nx.selfloop_edges(G_simple))

# Assign BFS-based 'layer' to each neuron starting from source nodes
source_nodes = [n for n in G_simple.nodes() if G_simple.in_degree(n) == 0]
if not source_nodes:
    source_nodes = [max(dict(G_simple.out_degree()), key=lambda n: G_simple.out_degree(n))]

layers = {}
for source in source_nodes:
    for n, dist in nx.single_source_shortest_path_length(G_simple, source).items():
        layers[n] = max(layers.get(n, 0), dist)

# Fill in any unassigned
for n in G_simple.nodes():
    if n not in layers:
        layers[n] = 0

max_layer = max(layers.values())
layer_groups = {}
for n, l in layers.items():
    layer_groups.setdefault(l, []).append(n)

print('BFS Layer Assignment:')
for l in sorted(layer_groups.keys()):
    print(f'  Layer {l}: neurons {sorted(layer_groups[l])}')

# Export the adjacency matrix and layer info
connectome_arch = {
    'adjacency_matrix' : A,
    'transition_matrix': P,
    'influence_matrix' : influence,
    'layers'           : layers,
    'layer_groups'     : layer_groups,
    'n_neurons'        : N,
    'n_connections'    : G.number_of_edges(),
    'nodes'            : nodes,
    'G'                : G,
}

with open(DATA_DIR / 'connectome_architecture.pkl', 'wb') as f:
    pickle.dump(connectome_arch, f)

print('\nConnectome architecture saved for Notebook 5.')

---

## Student Exercises

**Exercise 4.1** — The adjacency matrix $A$ is generally asymmetric for a directed connectome. What does symmetry in $A$ mean biologically? Compute $\frac{1}{2}(A + A^T)$ and compare its heatmap to the original.

**Exercise 4.2** — Compute the **eigenvalue spectrum** of the adjacency matrix. The largest eigenvalue (spectral radius) is related to the network's propensity to amplify signals. How does it compare between the detected and ground-truth connectomes?

**Exercise 4.3** — Implement a **leaky integrate-and-fire** model using the transition matrix `P`: at each step, each neuron's activation is `act_new[i] = decay * act[i] + sum_j P[j,i] * act[j]` where `decay=0.8`. Simulate for 20 steps starting from a unit pulse on the highest out-degree neuron. Does the network reach a steady state?

**Exercise 4.4** — Use `nx.community.louvain_communities(G.to_undirected())` to detect functional modules. Visualize the graph with nodes colored by community membership. Are neurons in the same community spatially close in the image?

**Exercise 4.5 (Challenge)** — Compare your detected connectome against the C. elegans connectome from the `connectome-tools` Python package or a CSV file of the 302-neuron connectome (available online). How do the degree distributions compare? What does this suggest about how neuronal circuits scale?